# Sunoco Weekly Gasoline Demand — Interaction Terms & Multiple Linear Regression

**Goal.** Identify which *interaction* (product) terms genuinely improve the weekly
`Demand` model, fit the resulting multiple linear regressions, validate them
out-of-sample, and emit ready-to-paste **Excel formulas**.

**This notebook adapts to your workbook.** Different versions of the Sunoco file carry
different covariates — some have a `major_holiday_week` flag, others carry
population-weighted degree days (`hdd_pop_weighted`, `cdd_pop_weighted`) instead. Rather
than hardcoding a column layout, section 1 detects what is present, and everything
downstream — the interaction pool, the models, the Excel column letters — follows from
that. Nothing below needs editing if your sheet differs.

| Section | What it does |
|---|---|
| 1 | Load, detect available predictors, derive Excel column letters |
| 2 | Build the interaction pool from what's actually present |
| 3 | Screen each interaction against a no-interaction baseline |
| 4 | Mean-centering — the fix for interaction-induced multicollinearity |
| 5 | Forward stepwise over the pool |
| 6 | Out-of-sample validation (this picks the final model) |
| 7 | Final model + regression equation |
| 8 | Auto-generated, self-verifying Excel formulas |
| 9 | Diagnostics and caveats |

In [ ]:
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

WORKBOOK = "Sunoco Training_complete_NEW.xlsx"  # <-- edit path if needed
SHEET = "Target"
MIN_COVERAGE = 0.95  # a predictor must be present in >=95% of weeks to be usable

## 1. Load, detect predictors, derive Excel column letters

Three things happen here that make the rest of the notebook portable:

1. **Headers are normalised** — coerced to string and stripped, so a stray trailing
   space from an Excel round-trip doesn't cause a `KeyError`.
2. **Excel column letters are read from the sheet**, not assumed. This is what keeps the
   generated formulas pointing at the right columns when your layout differs.
3. **Predictors are chosen by coverage**, so sparse columns can't silently gut the sample.

In [ ]:
df = pd.read_excel(WORKBOOK, sheet_name=SHEET)
df.columns = [str(c).strip() for c in df.columns]
df = df.loc[:, ~pd.Index(df.columns).str.startswith("Unnamed")]

# Read the header row directly so each variable maps to its true spreadsheet letter.
_ws = load_workbook(WORKBOOK, read_only=True)[SHEET]
_header = next(_ws.iter_rows(max_row=1, values_only=True))
EXCEL_COLUMNS = {
    str(name).strip(): get_column_letter(i)
    for i, name in enumerate(_header, start=1)
    if name is not None and str(name).strip()
}

for required in ("Demand", "Date"):
    if required not in df.columns:
        raise KeyError(f"'{required}' not found in sheet '{SHEET}'. Columns: {list(df.columns)}")

print(f"rows: {len(df)}   dates: {df['Date'].min():%Y-%m-%d} → {df['Date'].max():%Y-%m-%d}")
print(f"Demand observed: {df['Demand'].notna().sum()}   held-out: {df['Demand'].isna().sum()}")
print(f"columns detected: {len(df.columns)}")

In [ ]:
coverage = (df.notna().sum() / len(df)).sort_values(ascending=False)
report = pd.DataFrame({"coverage_%": (coverage * 100).round(1)})
report["excel_col"] = [EXCEL_COLUMNS.get(c, "—") for c in report.index]
report["usable"] = coverage >= MIN_COVERAGE
print(report.to_string())

### Choosing the predictor set

`PREFERRED` lists candidates in rough order of economic relevance. Only those actually
present and adequately covered survive into `BASE`. Add or remove names here if you want
a different starting set — everything downstream adapts.

In [ ]:
PREFERRED = [
    "regular_gas_usd_per_gal",      # price
    "vehicle_miles_mil",            # travel volume
    "refinery_utilization_pct",     # supply-side
    "us_population_thous",          # scale / slow trend
    "initial_jobless_claims",       # labour-market stress
    "rotary_rig_count",             # upstream activity
    "vehicle_sales_saar_mil",       # fleet turnover
    "cdd_pop_weighted",             # cooling degree days — summer driving season
    "hdd_pop_weighted",             # heating degree days — winter
    "major_holiday_week",           # 0/1 indicator
]

BASE = [c for c in PREFERRED if c in df.columns and coverage[c] >= MIN_COVERAGE]
SKIPPED = [c for c in PREFERRED if c in df.columns and coverage[c] < MIN_COVERAGE]
ABSENT = [c for c in PREFERRED if c not in df.columns]

print("USING   :", BASE)
print("SKIPPED (low coverage):", SKIPPED or "none")
print("ABSENT  (not in sheet):", ABSENT or "none")

train_full = df[df["Demand"].notna()].dropna(subset=BASE).copy()
train_full["year"] = train_full["Date"].dt.year
print(f"\ncomplete-case training rows: {len(train_full)} of {df['Demand'].notna().sum()} observed weeks")

if len(train_full) < 300:
    print("WARNING: small complete-case sample. Consider lowering MIN_COVERAGE or trimming PREFERRED.")

In [ ]:
print(train_full[BASE + ["Demand"]].corr()["Demand"].sort_values(ascending=False).round(3).to_string())

### Why interactions are worth trying

Check the sign on `regular_gas_usd_per_gal` above. If it is **positive**, that
contradicts the workbook's own hypothesis sheet, which predicts a strong negative price
effect — and it is a symptom, not a finding. Over 35 years nominal gas prices and
population/VMT all trended upward together, so a single linear price slope is forced to
do two jobs: capture long-run drift *and* the short-run demand response. An interaction
that lets the price slope vary by era separates them.

## 2. Building the interaction pool

Each candidate is a product of two columns. Rationale matters more than fit — an
interaction you cannot explain will not survive on new data.

| Term | Reads as | Rationale |
|---|---|---|
| `gas × population` | price slope drifts as the country grows | Both trend; lets the price response differ between the 1990s and 2020s instead of averaging them |
| `VMT × population` | miles per person, not total | Turns a raw-volume term into per-capita intensity |
| `gas × VMT` | price bites harder when driving is high | Elasticity scales with exposure |
| `rigs × gas` | supply- vs demand-driven price moves | High rigs *and* high prices signals a supply shock, which behaves differently |
| `gas × jobless claims` | price sensitivity under labour stress | Stressed households cut discretionary driving faster |
| `gas × vehicle sales` | fleet efficiency moderates price | New-vehicle mix shifts consumption per mile |
| `CDD × VMT` | summer driving amplification | Hot weeks and high mileage compound — the classic summer demand peak |
| `CDD × gas` | summer trips are less price-elastic | Vacation travel is committed in advance |
| `HDD × VMT` | cold suppresses discretionary trips | Winter mileage converts to fuel differently (cold-start penalty) |
| `VMT × holiday` | holiday bump scales with the travel base | A holiday on a 200bn-mile base ≠ on a 290bn-mile base |
| `gas × holiday` | holiday travel is less price-elastic | Thanksgiving trips happen regardless of pump price |

**One judgement call to flag:** `gas × population` is partly a proxy for time. Population
is nearly a linear trend, so the term doubles as "the price relationship has drifted."
That is real, but if you would rather model it explicitly, add a year index to `BASE` and
pair it with gas instead. If your sheet has degree days, they are a *better* seasonality
control than anything population can proxy.

In [ ]:
CANDIDATE_PAIRS = [
    ("regular_gas_usd_per_gal", "us_population_thous"),
    ("vehicle_miles_mil", "us_population_thous"),
    ("regular_gas_usd_per_gal", "vehicle_miles_mil"),
    ("rotary_rig_count", "regular_gas_usd_per_gal"),
    ("regular_gas_usd_per_gal", "initial_jobless_claims"),
    ("regular_gas_usd_per_gal", "vehicle_sales_saar_mil"),
    ("cdd_pop_weighted", "vehicle_miles_mil"),
    ("cdd_pop_weighted", "regular_gas_usd_per_gal"),
    ("hdd_pop_weighted", "vehicle_miles_mil"),
    ("vehicle_miles_mil", "major_holiday_week"),
    ("regular_gas_usd_per_gal", "major_holiday_week"),
]

SHORT = {
    "regular_gas_usd_per_gal": "gas",
    "us_population_thous": "pop",
    "vehicle_miles_mil": "vmiles",
    "rotary_rig_count": "rigs",
    "initial_jobless_claims": "jobless",
    "vehicle_sales_saar_mil": "vsales",
    "major_holiday_week": "holiday",
    "cdd_pop_weighted": "cdd",
    "hdd_pop_weighted": "hdd",
    "refinery_utilization_pct": "refutil",
}

INTERACTIONS = {
    f"{SHORT.get(a, a)}_x_{SHORT.get(b, b)}": (a, b)
    for a, b in CANDIDATE_PAIRS
    if a in BASE and b in BASE
}
DROPPED = [(a, b) for a, b in CANDIDATE_PAIRS if a not in BASE or b not in BASE]

print(f"interaction pool ({len(INTERACTIONS)}):")
for k, (a, b) in INTERACTIONS.items():
    print(f"  {k:<22} = {a} × {b}")
print(f"\nunavailable in this workbook ({len(DROPPED)}): {[f'{a}×{b}' for a, b in DROPPED] or 'none'}")


def add_interactions(data, terms, means=None):
    """Mean-center the base predictors, then form interaction products.

    Passing `means` from the training set keeps validation honest — validation rows
    must be centered on training means, never on their own.
    """
    if means is None:
        means = data[BASE].mean()
    X = data[BASE] - means
    for name, (a, b) in terms.items():
        X[name] = X[a] * X[b]
    return X, means


def fit_ols(y, X):
    return sm.OLS(y, sm.add_constant(X)).fit()

## 3. Screen each interaction against a baseline

Baseline = main effects only. Each candidate is added one at a time. **ΔAIC below −2** is
the conventional bar for "this term earns its degree of freedom"; more negative is better.

In [ ]:
X_base, MEANS = add_interactions(train_full, {})
model_base = fit_ols(train_full["Demand"], X_base)
print(f"BASELINE   R² = {model_base.rsquared:.4f}   adj R² = {model_base.rsquared_adj:.4f}   AIC = {model_base.aic:.1f}\n")

screen_rows = []
for name, pair in INTERACTIONS.items():
    Xi, _ = add_interactions(train_full, {name: pair})
    mi = fit_ols(train_full["Demand"], Xi)
    screen_rows.append(
        {
            "interaction": name,
            "adj_R2": round(mi.rsquared_adj, 4),
            "delta_AIC": round(mi.aic - model_base.aic, 1),
            "coefficient": mi.params[name],
            "p_value": mi.pvalues[name],
        }
    )

screen = pd.DataFrame(screen_rows).sort_values("delta_AIC").reset_index(drop=True)
print(screen.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

## 4. Centering: the fix for interaction-induced multicollinearity

The step most commonly skipped, and it matters. A product of two large positive numbers
correlates almost perfectly with its own parents. The **fit is identical either way** —
but uncentered, you get VIFs in the thousands, uninterpretable standard errors, and
coefficients that flip sign on small data changes.

In [ ]:
def vif_table(X):
    Xc = sm.add_constant(X)
    return pd.DataFrame(
        {"variable": Xc.columns, "VIF": [variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])]}
    )


demo_keys = list(screen["interaction"].head(3))
demo_terms = {k: INTERACTIONS[k] for k in demo_keys}
print(f"demonstrating with the top-3 screened terms: {demo_keys}\n")

X_raw = train_full[BASE].copy()
for nm, (a, b) in demo_terms.items():
    X_raw[nm] = X_raw[a] * X_raw[b]
X_cen, _ = add_interactions(train_full, demo_terms)

print(
    vif_table(X_raw)
    .merge(vif_table(X_cen), on="variable", suffixes=("_uncentered", "_centered"))
    .round(1)
    .to_string(index=False)
)
print(
    f"\nR² uncentered = {fit_ols(train_full['Demand'], X_raw).rsquared:.6f}"
    f"   R² centered = {fit_ols(train_full['Demand'], X_cen).rsquared:.6f}   (identical, as expected)"
)

Everything below uses centered predictors. One consequence when reading coefficients: a
main effect is now **the slope at the average value of the other variable**, not at zero,
and the intercept is predicted demand with every predictor at its mean.

## 5. Forward stepwise over the interaction pool

Main effects stay locked in; only interactions compete. Stop when no remaining term
improves AIC by more than 2.

In [ ]:
selected, remaining = [], list(INTERACTIONS)
best = model_base
print(f"start                        AIC = {best.aic:.1f}")

while remaining:
    trials = []
    for cand in remaining:
        terms = {k: INTERACTIONS[k] for k in selected + [cand]}
        Xt, _ = add_interactions(train_full, terms)
        trials.append((fit_ols(train_full["Demand"], Xt), cand))
    trials.sort(key=lambda t: t[0].aic)
    top_model, top_name = trials[0]
    if top_model.aic >= best.aic - 2:
        break
    best = top_model
    selected.append(top_name)
    remaining.remove(top_name)
    print(f"  + {top_name:<22} AIC = {best.aic:8.1f}   adj R² = {best.rsquared_adj:.4f}")

print(f"\nstepwise selection ({len(selected)}): {selected}")

## 6. Out-of-sample validation — this is what decides the final model

In-sample AIC rewards complexity. Since the point of this workbook is forecasting
held-out years, the model must be judged on data it did not see. Train on the earlier
years, validate on the last three observed.

This deliberately keeps **2020 inside training** — the COVID collapse is the most
influential period in the series, and a model that can't survive learning from it won't
survive the forecast either.

In [ ]:
split_year = int(train_full["year"].max()) - 2
train = train_full[train_full["year"] < split_year]
valid = train_full[train_full["year"] >= split_year]
print(f"train {len(train)} rows (<{split_year})   validate {len(valid)} rows (≥{split_year})\n")

top_screened = list(screen["interaction"].head(3))
CANDIDATE_MODELS = {
    "0. no interactions": [],
    "1. stepwise winner": selected,
    "2. top-3 screened": top_screened,
    "3. stepwise first half": selected[: max(1, len(selected) // 2)],
    "4. all candidates": list(INTERACTIONS),
}

results = []
for label, keys in CANDIDATE_MODELS.items():
    terms = {k: INTERACTIONS[k] for k in keys}
    Xt, mu = add_interactions(train, terms)
    Xv, _ = add_interactions(valid, terms, means=mu)
    m = fit_ols(train["Demand"], Xt)
    err = valid["Demand"] - m.predict(sm.add_constant(Xv, has_constant="add"))
    results.append(
        {
            "model": label,
            "k": len(keys),
            "in_sample_R2": round(m.rsquared, 4),
            "OOS_RMSE": round(float(np.sqrt((err ** 2).mean())), 1),
            "OOS_MAPE_%": round(float((err.abs() / valid["Demand"]).mean() * 100), 2),
        }
    )

leaderboard = pd.DataFrame(results).sort_values("OOS_RMSE").reset_index(drop=True)
print(leaderboard.to_string(index=False))

**Watch for the model with the most interactions losing here.** Adding every term that
looks significant in-sample buys R² and pays for it on new data. If a smaller model wins
on RMSE, that is the one to ship — and this table is the argument if anyone asks why you
dropped significant terms.

The winner is selected automatically below.

In [ ]:
FINAL_INTERACTIONS = CANDIDATE_MODELS[leaderboard.iloc[0]["model"]]
print(f"selected by OOS RMSE: {leaderboard.iloc[0]['model']} → {FINAL_INTERACTIONS or 'no interactions'}")

## 7. Final model, refit on all training data

Having chosen the specification on the holdout, refit on every observed week so the
coefficients going into Excel use all available information.

In [ ]:
FINAL_TERMS = {k: INTERACTIONS[k] for k in FINAL_INTERACTIONS}
X_final, FINAL_MEANS = add_interactions(train_full, FINAL_TERMS)
final_model = fit_ols(train_full["Demand"], X_final)
print(final_model.summary())

In [ ]:
print("Centering constants (needed for the centered Excel formula):\n")
print(FINAL_MEANS.round(6).to_string())


def render_equation(model, means):
    lines = [f"Demand_hat = {model.params['const']:.6f}"]
    for term, coef in model.params.items():
        if term == "const":
            continue
        if term in INTERACTIONS:
            a, b = INTERACTIONS[term]
            piece = f"({a} - {means[a]:.4f}) * ({b} - {means[b]:.4f})"
        else:
            piece = f"({term} - {means[term]:.4f})"
        lines.append(f"    {'+' if coef >= 0 else '-'} {abs(coef):.8f} * {piece}")
    return "\n".join(lines)


print("\n" + render_equation(final_model, FINAL_MEANS))

### Marginal effects — reading the interactions

With interactions, the effect of gas price is no longer one number; it depends on the
variables it interacts with. This is the payoff: if the pooled model reported a positive
price coefficient, check whether it turns negative at modern conditions once the slope is
allowed to vary.

In [ ]:
b = final_model.params
gas = "regular_gas_usd_per_gal"
gas_partners = {k: v for k, v in FINAL_TERMS.items() if gas in v}

if gas in BASE and gas_partners:
    rows = []
    for label, q in [("low (10th pct)", 0.10), ("sample average", None), ("high (90th pct)", 0.90)]:
        slope = b[gas]
        detail = {}
        for term, (p, p2) in gas_partners.items():
            other = p2 if p == gas else p
            val = FINAL_MEANS[other] if q is None else train_full[other].quantile(q)
            slope += b[term] * (val - FINAL_MEANS[other])
            detail[other] = round(float(val), 1)
        rows.append({"scenario": label, **detail, "d_Demand_per_$1_gas": round(float(slope), 1)})
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("No gas-price interactions in the final model; the price slope is constant:")
    print(f"  d(Demand)/d($1 gas) = {b.get(gas, float('nan')):.1f}")

### Sanity check: can this model predict negative demand?

The expanded Excel formula in section 8 may show a large negative intercept. That is an
algebraic artifact of un-centering — it is the value when *every* predictor is literally
zero (zero population, zero gas price), a row that cannot exist. The check below confirms
predictions stay in a sane range across both observed and held-out weeks.

In [ ]:
fitted = final_model.predict(sm.add_constant(X_final))
print(f"fitted    : min {fitted.min():,.0f}   max {fitted.max():,.0f}   (actual {train_full['Demand'].min():,.0f}–{train_full['Demand'].max():,.0f})")
print(f"fitted <= 0: {(fitted <= 0).sum()} of {len(fitted)}")

holdout = df[df["Demand"].isna()].dropna(subset=BASE).copy()
if len(holdout):
    Xh, _ = add_interactions(holdout, FINAL_TERMS, means=FINAL_MEANS)
    pred_h = final_model.predict(sm.add_constant(Xh, has_constant="add"))
    print(f"\nforecast  : min {pred_h.min():,.0f}   max {pred_h.max():,.0f}   over {len(pred_h)} held-out weeks")
    print(f"forecast <= 0: {(pred_h <= 0).sum()}")

## 8. Excel formulas

Column letters were read from your sheet's header row in section 1, so these point at the
right columns even though workbook versions differ. Row 2 is assumed to be the first data
row; references are relative, so you can fill down.

In [ ]:
print("variable → Excel column (from your sheet):")
for v in BASE:
    print(f"  {v:<32} {EXCEL_COLUMNS[v]}")


def excel_centered(model, means, row=2, decimals=10):
    """Centered form — mirrors the notebook maths exactly. Verbose but transparent."""
    parts = [f"={model.params['const']:.{decimals}f}"]
    for term, coef in model.params.items():
        if term == "const":
            continue
        sign, mag = ("+" if coef >= 0 else "-"), f"{abs(coef):.{decimals}f}"
        if term in INTERACTIONS:
            a, bb = INTERACTIONS[term]
            parts.append(
                f"{sign}{mag}*({EXCEL_COLUMNS[a]}{row}-{means[a]:.10f})*({EXCEL_COLUMNS[bb]}{row}-{means[bb]:.10f})"
            )
        else:
            parts.append(f"{sign}{mag}*({EXCEL_COLUMNS[term]}{row}-{means[term]:.10f})")
    return "".join(parts)


def excel_expanded(model, means, row=2, decimals=10):
    """Algebraically expanded to raw columns — shorter, no centering constants.

    Expanding (a-ma)(b-mb) = ab - ma*b - mb*a + ma*mb redistributes each interaction
    into the main effects and the intercept. Identical predictions to the centered form.
    """
    intercept = model.params["const"]
    linear = {v: 0.0 for v in BASE}
    products = {}
    for term, coef in model.params.items():
        if term == "const":
            continue
        if term in INTERACTIONS:
            a, bb = INTERACTIONS[term]
            products[(a, bb)] = products.get((a, bb), 0.0) + coef
            linear[a] -= coef * means[bb]
            linear[bb] -= coef * means[a]
            intercept += coef * means[a] * means[bb]
        else:
            linear[term] += coef
            intercept -= coef * means[term]

    parts = [f"={intercept:.{decimals}f}"]
    for v, coef in linear.items():
        if abs(coef) > 1e-12:
            parts.append(f"{'+' if coef >= 0 else '-'}{abs(coef):.{decimals}f}*{EXCEL_COLUMNS[v]}{row}")
    for (a, bb), coef in products.items():
        parts.append(
            f"{'+' if coef >= 0 else '-'}{abs(coef):.{decimals}f}*{EXCEL_COLUMNS[a]}{row}*{EXCEL_COLUMNS[bb]}{row}"
        )
    return "".join(parts)


formula_centered = excel_centered(final_model, FINAL_MEANS)
formula_expanded = excel_expanded(final_model, FINAL_MEANS)

print("\nCENTERED FORM (paste into a new column, row 2, then fill down)\n")
print(formula_centered)
print("\n" + "-" * 100 + "\nEXPANDED FORM (equivalent, raw columns only)\n")
print(formula_expanded)

### Verify the formulas reproduce the Python predictions

Never paste a generated formula without checking it. This evaluates both strings against
the model's own `.predict()` output.

In [ ]:
def evaluate_formula(formula, data, row=2):
    expr = formula.lstrip("=")
    used = {v: EXCEL_COLUMNS[v] for v in BASE}
    # Longest letters first so "AA" is not partially matched by "A".
    for var in sorted(used, key=lambda v: len(used[v]), reverse=True):
        expr = expr.replace(f"{used[var]}{row}", f"data['{var}']")
    return eval(expr)  # noqa: S307 — evaluating our own generated string


check = pd.DataFrame(
    {
        "python": fitted,
        "excel_centered": evaluate_formula(formula_centered, train_full),
        "excel_expanded": evaluate_formula(formula_expanded, train_full),
    }
)
print(check.head().round(4).to_string())
d_cen = (check.python - check.excel_centered).abs().max()
d_exp = (check.python - check.excel_expanded).abs().max()
print(f"\nmax |python − centered| = {d_cen:.3e}\nmax |python − expanded| = {d_exp:.3e}")
print("\nBoth forms reproduce the model." if max(d_cen, d_exp) < 1e-3 else "\nMISMATCH — do not paste.")

### Formulas for every model tested

Drop any of these into the sheet to compare side by side.

In [ ]:
for label, keys in CANDIDATE_MODELS.items():
    terms = {k: INTERACTIONS[k] for k in keys}
    Xm, mu = add_interactions(train_full, terms)
    m = fit_ols(train_full["Demand"], Xm)
    print(f"\n### {label}   (adj R² = {m.rsquared_adj:.4f})\n")
    print(excel_expanded(m, mu))

### Optional: write predictions back into the workbook

In [ ]:
def write_predictions(path_in, path_out, model, means, header="Interaction Model"):
    """Append the model as a new formula column. Regenerated per row, so digits
    inside coefficients are never mangled by string replacement."""
    wb = load_workbook(path_in)
    ws = wb[SHEET]
    col = ws.max_column + 1
    ws.cell(row=1, column=col, value=header)
    for r in range(2, ws.max_row + 1):
        ws.cell(row=r, column=col, value=excel_expanded(model, means, row=r))
    wb.save(path_out)
    return path_out


# write_predictions(WORKBOOK, "Sunoco_with_interactions.xlsx", final_model, FINAL_MEANS)

## 9. Diagnostics and caveats

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

resid = final_model.resid
_, bp_p, _, _ = het_breuschpagan(resid, sm.add_constant(X_final))
print(f"Durbin–Watson         : {durbin_watson(resid):.3f}   (2.0 = no autocorrelation)")
print(f"Breusch–Pagan p-value : {bp_p:.3e}   (< 0.05 ⇒ heteroskedastic)")
print(f"Residual std. error   : {np.sqrt(final_model.mse_resid):.1f} thousand barrels/day")
print(f"Skew {resid.skew():.3f}   Kurtosis {resid.kurtosis():.3f}")

In [ ]:
worst = train_full.assign(residual=resid).reindex(resid.abs().sort_values(ascending=False).index).head(10)
print(worst[["Date", "Demand", "residual"] + BASE[:3]].round(1).to_string(index=False))

### What to be careful about

1. **Autocorrelation is the main problem.** Durbin–Watson well below 2 means consecutive
   weeks' errors are related — expected for a weekly series with no lag terms.
   Coefficients stay unbiased, but the standard errors and p-values above are **too
   small**, so significance is overstated. Fixes in ascending order of effort:
   Newey–West errors (`final_model.get_robustcov_results(cov_type="HAC", maxlags=8)`),
   adding `Demand.shift(1)` as a predictor, or a proper time-series model. Note a lagged
   term is unavailable for multi-week-ahead forecasts unless you forecast recursively.

2. **Seasonality.** If your workbook has `hdd_pop_weighted` / `cdd_pop_weighted`, you
   already have strong seasonality controls and should keep them. If not, add month
   dummies or a week-of-year Fourier pair — likely a bigger gain than any interaction here.

3. **`gas × population` is partly a time proxy.** Interpret it as "the price relationship
   has drifted," not as a causal population mechanism.

4. **Nominal prices, not real.** Gas price is not inflation-adjusted across 35 years.
   Deflating by CPI is cleaner than asking an interaction to absorb the drift.

5. **Forecasts extrapolate.** Population in the held-out years exceeds anything in
   training, and interaction terms extrapolate faster than linear ones because the product
   grows faster than either parent. Trust the near horizon more than the far end.